[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/Intelligence-Artificielle-et-Data-Science/blob/main/bloc2_donnees/corrections/seance1_correction.ipynb)

# Séance 2.1 — Charger, comprendre et nettoyer un jeu de données

**Correction** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/Intelligence-Artificielle-et-Data-Science/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- dire en 30 secondes ce que contient un fichier que vous n'avez jamais vu
- sélectionner les lignes et les colonnes qui vous intéressent, et calculer sur une colonne entière
- repérer les défauts classiques d'un fichier réel : doublons, trous, types faux
- convertir du texte en nombres et en dates — et vérifier que la conversion dit vrai
- annoncer combien de lignes votre nettoyage a fait perdre, et pourquoi

## Correction

Solutions commentées. Comparez avec ce que vous aviez écrit : plusieurs formulations peuvent être correctes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/Intelligence-Artificielle-et-Data-Science/main/bloc2_donnees/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
ventes = pd.read_csv(BASE + "ventes.csv")
clients = pd.read_csv(BASE + "clients.csv")
produits = pd.read_csv(BASE + "produits.csv")
sale = pd.read_csv(BASE + "ventes_sale.csv")   ## le fichier brut, a nettoyer

print(ventes.shape, clients.shape, produits.shape, sale.shape)

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — Combien de produits au catalogue ?

> **Votre mission :**
> - Charger `produits.csv` dans une variable `produits`.
> - Mettre le nombre de lignes dans `nb_produits`.

In [ ]:
produits = pd.read_csv(BASE + "produits.csv")

# .shape renvoie (lignes, colonnes) : la position 0 est le nombre de lignes
nb_produits = produits.shape[0]   ## [1] donnerait le nombre de colonnes

print(nb_produits)

In [ ]:
verifier("1 - nombre de produits", nb_produits == 2956,
         "shape renvoie un couple : la position 0 est le nombre de lignes")

### Exercice 2 — Choisir des colonnes, choisir des lignes

> **Votre mission :**
> - Construire `apercu` avec **uniquement** les colonnes `date`, `qte` et `prix`. Attention aux doubles crochets.
> - Puis garder dans `grosses` les lignes où `qte` dépasse **strictement** 100, et mettre leur nombre dans `nb_grosses`.

In [ ]:
# Doubles crochets : les exterieurs selectionnent, les interieurs
# delimitent la liste des colonnes voulues
apercu = ventes[["date", "qte", "prix"]]   ## l'ordre donne est respecte

grosses = ventes.query("qte > 100")   ## strictement, donc > et non >=
nb_grosses = grosses.shape[0]         ## le nombre de lignes retenues

print(list(apercu.columns), "|", nb_grosses, "grosses lignes")

In [ ]:
verifier("2a - selection de colonnes",
         list(apercu.columns) == ["date", "qte", "prix"],
         "il faut une LISTE de colonnes, donc des doubles crochets")
verifier("2b - grosses commandes", nb_grosses == 560,
         "la condition va entre guillemets : query(\"qte > 100\")")

### Exercice 3 — Le chiffre d'affaires, et le prix typique

> **Votre mission :**
> - Créer la colonne `ca` = quantité × prix, puis mettre le total dans `ca_total`, arrondi à 2 décimales.
> - Calculer aussi le prix **moyen** dans `prix_moyen` et le prix **médian** dans `prix_median`, arrondis à 2 décimales.
> - Comparez les deux derniers. Lequel décrit le mieux « le produit typique » ?

In [ ]:
# Une seule instruction, et les 45 123 lignes sont calculees
ventes["ca"] = ventes["qte"] * ventes["prix"]   ## une colonne toute neuve
ca_total = round(ventes["ca"].sum(), 2)         ## .sum() additionne tout

prix_moyen = round(ventes["prix"].mean(), 2)     ## la moyenne
prix_median = round(ventes["prix"].median(), 2)  ## la valeur du milieu

# Moyenne deux fois plus haute que la mediane : quelques prix tres eleves
# tirent la moyenne vers le haut. C'est la mediane qui decrit le typique.

print(ca_total, "euros | moyenne :", prix_moyen, "| mediane :", prix_median)

In [ ]:
verifier("3a - CA total", ca_total == 1152913.87,
         "ca = qte * prix, puis .sum() sur la colonne entiere")
verifier("3b - prix moyen", prix_moyen == 3.93, "la methode s'appelle mean()")
verifier("3c - prix median", prix_median == 1.95, "la methode s'appelle median()")

### Exercice 4 — Dédoublonner, puis écarter les ventes sans client

> **Votre mission :**
> - On passe au fichier sale. Créer `net` : `sale` sans les lignes dupliquées.
> - ⚠️ On dédoublonne **avant tout le reste** : filtrer d'abord ferait perdre le compte de ce qu'on retire.
> - Puis retirer de `net` les lignes dont `client_id` est manquant — parce que notre question porte sur les **clients**.

In [ ]:
# On dedoublonne AVANT tout le reste : sinon les doublons se propagent
# dans toutes les etapes suivantes. .copy() evite les avertissements
# quand on modifiera net plus bas.
net = sale.drop_duplicates().copy()   ## 5370 -> 5119 lignes
nb_dedoublonne = len(net)             ## a noter AVANT l'etape suivante

# subset=["client_id"] : on ne supprime que si CETTE colonne est vide,
# pas des qu'une colonne quelconque a un trou
net = net.dropna(subset=["client_id"]).copy()   ## un dropna() nu en ferait plus

# Mesurer a chaque etape, c'est ce qui permettra d'ecrire le bilan
print(len(sale), "->", nb_dedoublonne, "->", len(net))

In [ ]:
verifier("4a - apres dedoublonnage", nb_dedoublonne == 5119,
         "5370 - 251 : la methode s'appelle drop_duplicates()")
verifier("4b - lignes avec client", len(net) == 4712,
         "dropna(subset=[...]) cible une colonne precise")

### Exercice 5 — Le prix en nombre

> **Votre mission :**
> - Enlever le suffixe ` EUR`, remplacer la virgule par un point, convertir en nombre.
> - Remettre le résultat dans `net["prix"]`.
> - Puis vérifier qu'aucune valeur n'a été perdue : `nb_prix_perdus`.

In [ ]:
# Etape 1 : enlever le suffixe " EUR"
txt = net["prix"].str.replace(" EUR", "", regex=False)   ## .str = du texte

# Etape 2 : virgule francaise -> point, que Python comprend
txt = txt.str.replace(",", ".", regex=False)

# Etape 3 : conversion. errors="coerce" met NaN au lieu de planter...
net["prix"] = pd.to_numeric(txt, errors="coerce")

# ... donc on verifie tout de suite combien de NaN ont ete crees
nb_prix_perdus = net["prix"].isna().sum()   ## doit valoir 0

print(net["prix"].dtype, "|", nb_prix_perdus, "valeurs perdues")

In [ ]:
verifier("5a - prix numerique", net["prix"].dtype == "float64",
         "pd.to_numeric convertit une colonne texte en nombres")
verifier("5b - aucune perte", nb_prix_perdus == 0,
         "si > 0, c'est qu'il reste du texte non converti dans la colonne")

### Exercice 6 — Les dates, correctement

> **Votre mission :**
> - Convertir `net["date"]` en vraies dates. Le fichier mélange `14/11/2011` et `24-11-2011`, et il est au format **français**.
> - Mettre la date la plus ancienne dans `date_min`.
> - Puis créer la colonne `mois` et mettre dans `mois_top` le numéro du mois qui compte le plus de lignes.

In [ ]:
# format="mixed" : deux ecritures cohabitent dans la colonne
# dayfirst=True : format francais, le jour avant le mois
net["date"] = pd.to_datetime(net["date"], format="mixed", dayfirst=True)
date_min = net["date"].min()   ## min() sur des dates : la plus ancienne

# .dt donne acces aux composants d'une colonne de dates
net["mois"] = net["date"].dt.month   ## un entier de 1 a 12

# idxmax() renvoie l'etiquette (le numero du mois), pas l'effectif
mois_top = net["mois"].value_counts().idxmax()   ## octobre

print(date_min, "| mois le plus charge :", mois_top)

In [ ]:
verifier("6a - date la plus ancienne", str(date_min)[:10] == "2010-12-01",
         "sans dayfirst=True, 01/12/2010 est lu comme le 12 janvier")
verifier("6b - mois le plus charge", mois_top == 10,
         ".dt.month sur une colonne de dates, puis value_counts().idxmax()")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Question 7 — La répartition, en pourcentage

> **Votre mission :**
> - Quelle **part** des clients chaque pays représente-t-il ? En %, arrondi à 1 décimale.
> - Un effectif brut ne se compare pas d'une année sur l'autre ; un pourcentage, si.
> - *Nouveau :* `value_counts(normalize=True)` renvoie des parts au lieu d'effectifs.

In [ ]:
part = clients["pays"].value_counts(normalize=True) * 100   ## en %

# Le Royaume-Uni pese la moitie du fichier clients a lui seul (49,8 %),
# et les trois premiers pays en font plus de 70 %.
part.round(1).head(5)

### Question 8 — La plus grosse ligne du fichier

> **Votre mission :**
> - Retrouver la **ligne entière** de `ventes` dont le chiffre d'affaires est le plus élevé. (La colonne `ca` a été créée à l'exercice 3.)
> - Regardez son `prod_id`, puis cherchez son libellé dans `produits`.
> - Est-ce vraiment un produit ?
> - *Rappel :* `idxmax()` donne l'étiquette de la ligne, `.loc[...]` va la chercher.

In [ ]:
ligne = ventes.loc[ventes["ca"].idxmax()]   ## idxmax = l'etiquette
print(ligne)

# prod_id vaut "M" : 4 161 EUR sur une seule ligne, et son libelle est
# "Manual" — une saisie manuelle au comptoir, pas un article du catalogue.
# Retenez-le : on y revient a la seance 2.2.
produits.query("prod_id == 'M'")

### Question 9 — Le diagnostic, et le prix d'un `dropna()` négligent

> **Votre mission :**
> - Calculer le **taux de valeurs manquantes de chaque colonne** de `sale`, en %, arrondi à 2 décimales. Combien de colonnes sont réellement touchées ?
> - Puis comparer le nombre de lignes que laissent un `dropna()` **sans argument** et un `dropna(subset=['client_id'])`.
> - Ici les deux donnent le même résultat. Dans quel cas seraient-ils très différents ?
> - *Nouveau :* sur des True/False, `.mean()` donne directement une proportion — `df.isna().mean()`.

In [ ]:
# isna() donne True/False, mean() en fait une proportion, x100 un %
print((sale.isna().mean() * 100).round(2))   ## un diagnostic en une ligne

print("depart              :", len(sale))
print("dropna() brut       :", len(sale.dropna()))   ## UN trou suffit a perdre
print("dropna(subset=...)  :", len(sale.dropna(subset=["client_id"])))

# Une seule colonne est touchee, client_id, a 7,99 % : c'est pour ca que
# les deux dropna donnent 4 941. Sur un fichier ou trois colonnes ont
# chacune 5 % de trous, le dropna() brut en supprimerait 15 %, puisqu'il
# suffit d'UN trou sur la ligne pour la perdre. Toujours nommer la colonne
# dont l'absence est redhibitoire.

### Question 10 — Regarder avant de convertir

> **Votre mission :**
> - Convertir le `prix` de `sale` en nombre avec `errors='coerce'`, en ne corrigeant **que** la virgule — et **sans écraser la colonne d'origine**.
> - Combien de valeurs échouent ? **Affichez quelques-unes des valeurs fautives d'origine.**
> - Ne jamais lancer un `coerce` sans avoir regardé ce qu'on s'apprête à transformer en `NaN`.

In [ ]:
essai = pd.to_numeric(sale["prix"].astype(str).str.replace(",", "."),
                      errors="coerce")

print(essai.isna().sum(), "valeurs refusent la conversion")
sale.loc[essai.isna(), "prix"].head(5)   ## REGARDER les fautives d'origine

# 798 valeurs, et ce ne sont pas des erreurs : c'est le suffixe " EUR".
# Un coerce aveugle les aurait mises a NaN, puis un dropna les aurait
# supprimees — 15 % du fichier perdu pour une unite ecrite en toutes
# lettres, et rien pour vous prevenir.

### Question 11 — La fenêtre d'observation

> **Votre mission :**
> - Convertir la colonne `date` de `sale` correctement, puis donner la date la plus ancienne, la plus récente, et le nombre de **jours** entre les deux.
> - Un rapport annuel calculé sur cette période serait-il honnête ?
> - *Nouveau :* une soustraction de dates donne une durée ; `.days` en extrait le nombre de jours.

In [ ]:
dates = pd.to_datetime(sale["date"], format="mixed", dayfirst=True)

print("du", dates.min().date(), "au", dates.max().date())
print((dates.max() - dates.min()).days, "jours")   ## une duree, en jours

# 373 jours : un peu plus d'un an, et surtout le fichier s'arrete le
# 9 decembre. Comparer decembre aux autres mois reviendrait a comparer
# neuf jours a trente. On y revient a la seance 2.2.

### Question 12 — Le compte rendu qualité

> **Votre mission :**
> - Vous rendez le fichier nettoyé. Produire les chiffres de la note d'accompagnement :
> - lignes au départ · lignes conservées · taux de perte en % · **nombre de ventes écartées** faute de client identifié, et la **part des ventes plausibles** qu'elles représentent · la **part du chiffre d'affaires réel** qu'elles emportent.
> - Les deux derniers sont ceux qu'on oublie, et ils ne servent qu'ensemble : comparez-les. Que vous dit l'écart — ou l'absence d'écart — entre les deux ?
> - ⚠️ Calculez ce CA sur les ventes **plausibles** uniquement. Les quantités à 99 999 produiraient sinon un total fictif qui écrase tout le reste.

In [ ]:
base = sale.drop_duplicates().copy()
base["prix"] = pd.to_numeric(base["prix"].astype(str)
                             .str.replace(",", ".").str.replace(" EUR", ""))
base["ca"] = base["qte"] * base["prix"]

# Les ventes plausibles : ni retours, ni saisies a 99 999
reel = base.query("qte > 0 and qte < 10000")
sans_client = reel[reel["client_id"].isna()]   ## celles qu'on ecarte
garde = reel.dropna(subset=["client_id"])      ## celles qu'on conserve

print(len(sale), "lignes au depart |", len(garde), "conservees")
print("perte de lignes :", round(100 * (1 - len(garde) / len(sale)), 1), "%")

# Les deux chiffres a comparer : la part des ventes plausibles qu'on ecarte,
# et la part du CA qu'elles emportent avec elles
part_lignes = 100 * len(sans_client) / len(reel)
part_ca = 100 * sans_client["ca"].sum() / reel["ca"].sum()

print("ventes ecartees :", len(sans_client), "lignes, soit",
      round(part_lignes, 1), "% des ventes plausibles")
print("CA ecarte       :", round(sans_client["ca"].sum(), 2), "euros, soit",
      round(part_ca, 1), "% du CA reel")

# Ces ventes sont bien reelles : on les ecarte seulement parce qu'on ne sait
# pas a QUI les attribuer. Elles pesent la meme part du CA que des lignes :
# ce sont des ventes de taille ORDINAIRE, pas un segment particulier — et
# c'est la comparaison des deux pourcentages qui le dit, pas le CA seul.
# Si l'ecart entre les deux avait ete grand, la perte aurait ete BIAISEE
# (on aurait jete surtout des grosses ventes, ou surtout des petites) et il
# aurait fallu le signaler dans la note.